# End-to-end electro-optical PIC Mach–Zehnder interferometer (MZI)

**Physical waveguide geometry → supermode splitting → coupler unitaries → input pre-phase → internal phase → composite unitary → photon probabilities.**

The **upper-rail input pre-phase** (χ) operates *before* DC1; the upper-rail **internal phase** (φ) lies *between* DC1 and DC2. Optical path order is **PRE → DC1 → EO → DC2**, so

$$U_{\rm PIC}=C_2 P_{\rm EO}(\phi) C_1 P_{\rm pre}(\chi),\quad P(q)=\operatorname{diag}(e^{iq},1).$$

**Model assumptions:** monochromatic, scalar 1D symmetric double-slab transverse eigenproblem, identical guides within each directional coupler, ideal lossless propagation and EO shifters; 2×2 single-photon path subspace only. This is a physically parameterized pedagogical device model, **not** a 3D fabrication design or a two-photon Fock-space simulator.

Change geometric and electrical controls to see each **component matrix**, **cumulative matrix**, state amplitudes, and measurement probabilities. The Hadamard preset computes exact 50:50 interaction lengths for the chosen slab geometry and sets χ=0, φ=−π/2. A separate free-space convention check compares the full complex unitary up to a global phase.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh_tridiagonal
from functools import lru_cache
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets

np.set_printoptions(precision=4, suppress=True)
I2 = np.eye(2, dtype=complex)

def coupler(theta):
    c,s = np.cos(theta),np.sin(theta)
    return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)

def phase(phi):
    return np.diag([np.exp(1j*phi), 1+0j])

def probabilities(psi):
    return np.abs(psi)**2

def pretty(M):
    return np.array2string(np.asarray(M), formatter={'complex_kind':lambda z:f'{z.real:+.4f}{z.imag:+.4f}j', 'float_kind':lambda z:f'{z:.5f}'})

@lru_cache(maxsize=512)
def supermode_beta(width_um, gap_um, n_core, n_clad, wavelength_um):
    """Two highest transverse eigenmodes of symmetric 1D double-slab scalar Helmholtz problem.

    [-d²/dx² - k0² n(x)²] E = -beta² E; boundaries E=0.
    Two highest betas correspond to symmetric and antisymmetric fundamental supermodes
    in the single-mode, weakly coupled parameter regime.
    """
    dx = 0.02  # micrometers
    halfspan = max(6.0, gap_um/2+width_um+3.0)
    x = np.arange(-halfspan, halfspan+dx/2, dx)
    radius = np.abs(x)
    n = np.where((radius >= gap_um/2) & (radius <= gap_um/2+width_um), n_core, n_clad)
    k0 = 2*np.pi/wavelength_um
    diagonal = 2/dx**2 - (k0*n[1:-1])**2
    offdiagonal = np.full(len(diagonal)-1, -1/dx**2)
    # Smallest two eigenvalues yield the largest two beta values.
    values = eigh_tridiagonal(diagonal, offdiagonal, select='i', select_range=(0,1), eigvals_only=True)
    if np.any(values>=0):
        raise ValueError('No two propagating supermodes for this model setting.')
    beta = np.sqrt(-values)
    beta_even,beta_odd = beta[0],beta[1]
    return beta_even,beta_odd

def stage_data(Ppre,C1,P,C2,psi0):
    elements=[I2,Ppre,C1,P,C2]
    names=['Input','Input pre-phase (upper rail)','Directional coupler 1','Internal EO phase (upper rail)','Directional coupler 2']
    T=I2.copy()
    result=[]
    for name,U in zip(names,elements):
        T=U@T
        psi=T@psi0
        result.append((name,U,T.copy(),psi.copy(),np.outer(psi,psi.conj())))
    return result


## Interactive component inspector

Individual matrices and cumulative products update when a control changes. Both phase settings are upper-rail phase shifts, with χ applied before DC1 and φ between DC1 and DC2. The Hadamard preset sets χ=0 and computes the coupler lengths from the **current geometry**. The input-phase slider can then test the complete free-space-to-PIC four-element convention.


In [ ]:
width=widgets.FloatSlider(value=0.75,min=0.3,max=1.3,step=0.05,description='Width (µm)',continuous_update=False,style={'description_width':'initial'})
gap1=widgets.FloatSlider(value=0.35,min=0.1,max=1.2,step=0.05,description='Gap DC1 (µm)',continuous_update=False,style={'description_width':'initial'})
gap2=widgets.FloatSlider(value=0.35,min=0.1,max=1.2,step=0.05,description='Gap DC2 (µm)',continuous_update=False,style={'description_width':'initial'})
length1=widgets.FloatSlider(value=15.26,min=0,max=50,step=0.0001,readout_format='.4f',description='Length DC1 (µm)',continuous_update=False,style={'description_width':'160px'},layout=widgets.Layout(width='680px'))
length2=widgets.FloatSlider(value=15.26,min=0,max=50,step=0.0001,readout_format='.4f',description='Length DC2 (µm)',continuous_update=False,style={'description_width':'160px'},layout=widgets.Layout(width='680px'))
core=widgets.FloatSlider(value=1.50,min=1.47,max=1.7,step=0.005,description='Core index',continuous_update=False,style={'description_width':'initial'})
clad=widgets.FloatSlider(value=1.45,min=1.4,max=1.46,step=0.005,description='Clad index',continuous_update=False,style={'description_width':'initial'})
lam=widgets.FloatSlider(value=1.55,min=1.2,max=1.65,step=0.01,description='Wavelength (µm)',continuous_update=False,style={'description_width':'initial'})
voltage=widgets.FloatSlider(value=-2.5,min=-10,max=10,step=0.25,description='Voltage (V)',continuous_update=False,style={'description_width':'initial'})
vpi=widgets.FloatSlider(value=5,min=1,max=10,step=0.5,description='Vπ (V)',continuous_update=False,style={'description_width':'initial'})
bias=widgets.FloatSlider(value=0,min=-np.pi,max=np.pi,step=np.pi/12,description='Bias phase (rad)',continuous_update=False,style={'description_width':'initial'})
prephase=widgets.FloatSlider(value=0,min=-np.pi,max=np.pi,step=np.pi/40,readout_format='.4f',description='Input pre-phase χ (rad)',continuous_update=False,style={'description_width':'initial'},layout=widgets.Layout(width='680px'))
showrho=widgets.Checkbox(value=False,description='Show density matrices')
input_state=widgets.Dropdown(options=[('|1,0⟩',0),('|0,1⟩',1)],value=0,description='Input')
out=widgets.Output()
refresh=widgets.Button(description='Update MZI',button_style='primary',icon='refresh')
preset=widgets.Button(description='Set exact Hadamard reference',button_style='success',icon='check')

controls=[width,gap1,gap2,length1,length2,core,clad,lam,voltage,vpi,bias,prephase,showrho,input_state]
def update(*_):
    with out:
        clear_output(wait=True)
        if core.value <= clad.value:
            print('Core refractive index must exceed cladding index.')
            return
        try:
            be1,bo1=supermode_beta(round(width.value,4),round(gap1.value,4),round(core.value,4),round(clad.value,4),round(lam.value,4))
            be2,bo2=supermode_beta(round(width.value,4),round(gap2.value,4),round(core.value,4),round(clad.value,4),round(lam.value,4))
            kap1,kap2=(be1-bo1)/2,(be2-bo2)/2
            th1,th2=kap1*length1.value,kap2*length2.value
            phi=bias.value+np.pi*voltage.value/vpi.value
            psi0=np.array([1,0],complex) if input_state.value==0 else np.array([0,1],complex)
            stages=stage_data(phase(prephase.value),coupler(th1),phase(phi),coupler(th2),psi0)
            print(f'Coupler 1: βeven={be1:.6f} µm⁻¹, βodd={bo1:.6f} µm⁻¹, κ={kap1:.6f} µm⁻¹, κL={th1:.4f} rad, cross={np.sin(th1)**2:.4f}')
            print(f'Coupler 2: βeven={be2:.6f} µm⁻¹, βodd={bo2:.6f} µm⁻¹, κ={kap2:.6f} µm⁻¹, κL={th2:.4f} rad, cross={np.sin(th2)**2:.4f}')
            print(f'Pre-phase: χ = {prephase.value:.5f} rad (upper input rail)')
            print(f'Internal EO phase: φ = {phi:.5f} rad = bias + π V / Vπ')
            Tfinal=stages[-1][2]
            H=np.array([[1,1],[1,-1]],dtype=complex)/np.sqrt(2)
            phase_overlap=np.vdot(H,Tfinal)/2 # e^{i gamma} if Tfinal=e^{i gamma}H
            gamma=np.angle(phase_overlap)
            corrected=np.exp(-1j*gamma)*Tfinal
            err=np.linalg.norm(corrected-H)
            print('\n========== FINAL COMPOSITE UNITARY ==========')
            print('U_MZI = C2 @ P_internal @ C1 @ P_pre =\n',pretty(Tfinal))
            print('Reference Hadamard H =\n',pretty(H))
            print(f'Best global phase gamma = {gamma:.8f} rad')
            print('Phase-corrected U_MZI =\n',pretty(corrected))
            print(f'Hadamard Frobenius error || exp(-i gamma) U_MZI - H || = {err:.6e}')
            print('Composite unitarity error || U† U − I ||_F =',f'{np.linalg.norm(Tfinal.conj().T@Tfinal-I2):.3e}')
            print('RESULT:', 'HADAMARD MATCH (up to global phase)' if err < 1e-8 else 'NOT a Hadamard match; adjust phase/coupler parameters')
            for k,(name,U,T,psi,rho) in enumerate(stages):
                print(f'\n--- Stage {k}: {name} ---')
                print('Element U =\n',pretty(U))
                print('Cumulative T = U @ preceding T =\n',pretty(T))
                print('State amplitudes [|1,0>, |0,1>] =',pretty(psi))
                print('Output probabilities [P10, P01] =', np.round(probabilities(psi),6))
                if showrho.value: print('Density matrix rho =\n',pretty(rho))
            fig,axes=plt.subplots(1,2,figsize=(11,3.5))
            x=np.arange(len(stages))
            vals=np.array([probabilities(s[3]) for s in stages])
            axes[0].plot(x,vals[:,0],'o-',label='P(1,0)')
            axes[0].plot(x,vals[:,1],'s-',label='P(0,1)')
            axes[0].set_xticks(x,['Input','Pre','DC1','EO','DC2']);axes[0].set_ylim(-0.05,1.05)
            axes[0].set_ylabel('Photon detection probability');axes[0].legend()
            amps=np.array([s[3] for s in stages]);
            axes[1].plot(x,np.angle(amps[:,0]),'o-',label='arg α')
            axes[1].plot(x,np.angle(amps[:,1]),'s-',label='arg β')
            axes[1].set_xticks(x,['Input','Pre','DC1','EO','DC2']);axes[1].set_ylabel('Amplitude phase (rad)');axes[1].legend()
            fig.tight_layout();plt.show()
            # Independent consistency tests
            for name,U,T,psi,rho in stages:
                assert np.allclose(U.conj().T@U,I2,atol=1e-12),name
                assert np.allclose(T.conj().T@T,I2,atol=1e-12),name
                assert np.isclose(np.linalg.norm(psi),1,atol=1e-12),name
                assert np.allclose(rho,T@np.outer(psi0,psi0.conj())@T.conj().T,atol=1e-12),name
            print('PASS: unitary element/cumulative matrices, normalized probabilities, density-matrix consistency.')
        except Exception as err:
            print('Model error:',repr(err))
for ctrl in controls: ctrl.observe(update,names='value')
refresh.on_click(update)
def set_hadamard(_):
    # Compute 50:50 lengths directly from the electromagnetic supermode solver.
    be1,bo1=supermode_beta(round(width.value,4),round(gap1.value,4),round(core.value,4),round(clad.value,4),round(lam.value,4))
    be2,bo2=supermode_beta(round(width.value,4),round(gap2.value,4),round(core.value,4),round(clad.value,4),round(lam.value,4))
    length1.value=float(np.pi/(4*((be1-bo1)/2)))
    length2.value=float(np.pi/(4*((be2-bo2)/2)))
    voltage.value=-vpi.value/2
    bias.value=0
    prephase.value=0
    update()
preset.on_click(set_hadamard)
display(widgets.VBox([widgets.HTML('<b>Waveguide cross-section and wavelength</b>'),width,core,clad,lam,
 widgets.HTML('<b>Upper-rail input pre-phase (before DC1)</b>'),prephase,
 widgets.HTML('<b>Directional coupler 1</b>'),gap1,length1,
 widgets.HTML('<b>Electro-optic modulator</b>'),voltage,vpi,bias,
 widgets.HTML('<b>Directional coupler 2</b>'),gap2,length2,
 widgets.HBox([input_state,showrho]),widgets.HBox([preset,refresh]),widgets.HTML('<small>The optical path is PRE → DC1 → EO → DC2. Coupler lengths are shown numerically at the right of each slider (µm). Click <b>Set exact Hadamard reference</b> to compute 50:50 lengths from current waveguide geometry and set the EO phase to −π/2. Any change automatically recalculates.</small>'),out]))
update()


## Numerical validation benchmarks

The benchmark cell below checks the component and composite unitaries, direct matrix multiplication against a closed-form independent calculation, optical stage ordering, preservation of photon probabilities under an input pre-phase for a basis input, phase sensitivity for a superposition input, an analytical interference fringe, the geometry-to-Hadamard reference, and a global-phase-insensitive matrix comparison. All tests use numerical assertions, not an output-probability-only comparison.

**Research extensions (not implemented):** polarization/frequency, detuned asymmetric guides, loss with environment modes, 2-photon Fock-state evolution, and full-vector waveguide solvers.


In [ ]:
# Executable, noninteractive regression suite. A failed criterion raises AssertionError.

def phase_aligned_distance(A, B):
    """Frobenius distance after optimal scalar global-phase alignment."""
    overlap = np.trace(B.conj().T @ A)
    if abs(overlap) < 1e-14:
        return np.linalg.norm(A-B)
    return np.linalg.norm(np.exp(-1j*np.angle(overlap))*A-B)

# 1. Unitarity and analytical directional-coupler splitting.
for theta in np.linspace(0, np.pi/2, 19):
    C = coupler(theta)
    assert np.allclose(C.conj().T@C, I2, atol=1e-13)
    assert np.isclose(abs((C@np.array([1.,0.]))[1])**2, np.sin(theta)**2, atol=1e-13)

# 2. Independent closed-form composite: C2 P(phi) C1 P(chi).
#    This formula is built directly from c_j,s_j,phase exponentials (not stage_data).
def analytic_mzi(t1,t2,phi,chi):
    c1,s1,c2,s2=np.cos(t1),np.sin(t1),np.cos(t2),np.sin(t2)
    p=np.exp(1j*phi); q=np.exp(1j*chi)
    return np.array([
        [q*(c2*c1*p-s2*s1), -1j*(c2*s1*p+s2*c1)],
        [-1j*q*(s2*c1*p+c2*s1), c2*c1-s2*s1*p]
    ],complex)
for t1,t2,phi,chi in [(0.1,0.4,0.7,-0.2),(np.pi/4,np.pi/4,-np.pi/2,0),(0.3,0.65,1.7,0.9)]:
    stages=stage_data(phase(chi),coupler(t1),phase(phi),coupler(t2),np.array([1,0],complex))
    expected=analytic_mzi(t1,t2,phi,chi)
    assert np.allclose(stages[-1][2],expected,atol=1e-13)
    for _,element,T,psi,rho in stages:
        assert np.allclose(element.conj().T@element,I2,atol=1e-12)
        assert np.allclose(T.conj().T@T,I2,atol=1e-12)
        assert np.isclose(np.sum(probabilities(psi)),1.,atol=1e-12)
        assert np.allclose(rho,np.outer(psi,psi.conj()),atol=1e-12)
    assert np.allclose(stages[1][2],phase(chi),atol=1e-13), 'Input pre-phase must be first'

# 3. For a basis input, input pre-phase is just a global factor; probabilities unchanged.
for chi in (-1.3, 0, 2.1):
    a=stage_data(phase(chi),coupler(.32),phase(.6),coupler(.59),np.array([1,0],complex))[-1][3]
    b=stage_data(I2,coupler(.32),phase(.6),coupler(.59),np.array([1,0],complex))[-1][3]
    assert np.allclose(abs(a)**2,abs(b)**2,atol=1e-12)
# For an input superposition, the same input phase generally changes output statistics.
psi_plus=np.array([1.,1.],complex)/np.sqrt(2)
result_a=stage_data(phase(0),coupler(.4),phase(.7),coupler(.6),psi_plus)[-1][3]
result_b=stage_data(phase(.8),coupler(.4),phase(.7),coupler(.6),psi_plus)[-1][3]
assert np.linalg.norm(abs(result_a)**2-abs(result_b)**2)>1e-3

# 4. Ideal balanced MZI fringe: P10=sin²(phi/2), P01=cos²(phi/2).
for phi in np.linspace(-np.pi,np.pi,25):
    psi=analytic_mzi(np.pi/4,np.pi/4,phi,0)@np.array([1,0],complex)
    assert np.allclose(abs(psi)**2,[np.sin(phi/2)**2,np.cos(phi/2)**2],atol=1e-12)

# 5. Physics-based coupler lengths and exact full-complex Hadamard benchmark.
params=(.75,.35,1.50,1.45,1.55)
be,bo=supermode_beta(*params)
kappa=(be-bo)/2
L50=np.pi/(4*kappa)
H=np.array([[1,1],[1,-1]],complex)/np.sqrt(2)
T=analytic_mzi(kappa*L50,kappa*L50,-np.pi/2,0)
expected_phase=np.exp(-3j*np.pi/4)
assert np.allclose(T,expected_phase*H,atol=2e-12)
assert phase_aligned_distance(T,H)<2e-12
assert np.allclose(T.conj().T@T,I2,atol=2e-12)
# Positive χ corresponds to a physical phase on the upper input rail.
chi=.43
assert np.allclose(analytic_mzi(np.pi/4,np.pi/4,-np.pi/2,chi),expected_phase*H@phase(chi),atol=2e-12)

# 6. Geometry sanity check within chosen model regime: larger gap -> smaller coupling.
be_small,bo_small=supermode_beta(.75,.25,1.50,1.45,1.55)
be_large,bo_large=supermode_beta(.75,.75,1.50,1.45,1.55)
assert (be_small-bo_small)/2 > (be_large-bo_large)/2 > 0

print('PASS: six benchmark groups; coupler and composite unitarity, stage ordering,')
print('      independent matrix product, pre-phase behavior, interference, Hadamard, and gap trend.')
print(f'Physical reference κ = {kappa:.9f} µm^-1; L_50:50 = {L50:.9f} µm')
print('Hadamard target H =\n',pretty(H))
print('Composite U = C2 @ P_internal @ C1 @ P_pre =\n',pretty(T))
print('Global-phase-aligned matrix residual =',phase_aligned_distance(T,H))
